# Crop Classification using Hyperspectral Imaging: Hybrid ViT + GCN / Graph Transformer
Pipeline (as in the deck): load HSI -> PCA (30) -> SLIC superpixel graph (KNN k=5) -> 70/30 stratified split (before patch extraction) -> 7x7 patches -> ViT branch (128-d) + GCN/Graph-Transformer branch (64-d per node) -> linear fusion -> class-weighted CE -> OA / AA / Kappa / per-class P-R-F1.

In [ ]:
# CELL 1: IMPORTS + CONFIG
import numpy as np
import scipy.io as sio
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.neighbors import kneighbors_graph
from sklearn.metrics import (accuracy_score, cohen_kappa_score, classification_report,
                             confusion_matrix, precision_recall_fscore_support)
from skimage.segmentation import slic

SEED        = 42
DATASET     = "salinas"      # "salinas" or "indian_pines" (only used for class names)
N_PCA       = 30
PATCH       = 7
N_SEGMENTS  = 200
KNN_K       = 5
COMPACTNESS = 0.1            # SLIC on [0,1]-scaled PCs; raise for squarer regions, lower for tighter-to-spectra
VIT_DIM     = 128
GRAPH_DIM   = 64
EPOCHS      = 15
LR          = 1e-3
BATCH_SIZE  = 64

np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

CLASS_NAMES = {
    "salinas": ["Brocoli_green_weeds_1", "Brocoli_green_weeds_2", "Fallow", "Fallow_rough_plow",
                "Fallow_smooth", "Stubble", "Celery", "Grapes_untrained", "Soil_vinyard_develop",
                "Corn_senesced_green_weeds", "Lettuce_romaine_4wk", "Lettuce_romaine_5wk",
                "Lettuce_romaine_6wk", "Lettuce_romaine_7wk", "Vinyard_untrained",
                "Vinyard_vertical_trellis"],
    "indian_pines": ["Alfalfa", "Corn-notill", "Corn-mintill", "Corn", "Grass-pasture", "Grass-trees",
                     "Grass-pasture-mowed", "Hay-windrowed", "Oats", "Soybean-notill",
                     "Soybean-mintill", "Soybean-clean", "Wheat", "Woods",
                     "Buildings-Grass-Trees-Drives", "Stone-Steel-Towers"],
}

In [ ]:
# CELL 2: DATASET LOADING (upload the .mat files: data + ground truth)
def load_dataset_auto():
    from google.colab import files
    uploaded = files.upload()
    hsi, gt = None, None

    for fname in uploaded.keys():
        if fname.endswith(".mat"):
            data = sio.loadmat(fname)
            for k, v in data.items():
                if k.startswith("__") or not isinstance(v, np.ndarray):
                    continue
                if v.ndim == 3 and hsi is None:          # (H, W, bands)
                    hsi = v
                elif v.ndim == 2 and "gt" in k.lower():  # (H, W) labels
                    gt = v
        elif fname.endswith(".npy"):
            arr = np.load(fname)
            if "gt" in fname.lower():
                gt = arr
            else:
                hsi = arr

    if hsi is None or gt is None:
        raise ValueError("Could not detect dataset. Upload both the HSI cube and the ground-truth file.")

    hsi = hsi.astype(np.float32)
    gt = gt.astype(np.int64)
    print("HSI shape:", hsi.shape, "| GT shape:", gt.shape)
    print("Classes:", int(gt.max()), "| Labelled pixels:", int((gt > 0).sum()))
    return hsi, gt

hsi, gt = load_dataset_auto()
H, W, C = hsi.shape
num_classes = int(gt.max())

In [ ]:
# CELL 3: PREPROCESSING (band standardisation + PCA -> 30 components)
class PCAReducer:
    def __init__(self, n_components=30):
        self.scaler = StandardScaler()
        self.pca = PCA(n_components=n_components, random_state=SEED)

    def fit_transform(self, X):
        H, W, C = X.shape
        flat = self.scaler.fit_transform(X.reshape(-1, C))
        pcs = self.pca.fit_transform(flat)
        pcs = pcs / pcs.std()                      # global scale, keeps relative PC variance
        return pcs.reshape(H, W, -1).astype(np.float32)

X_pca = PCAReducer(N_PCA).fit_transform(hsi)
print("PCA cube:", X_pca.shape)

In [ ]:
# CELL 4: SLIC SUPERPIXELS + KNN GRAPH (built once on the full image, uses no labels)
def build_graph(X, n_segments=200, k=5, compactness=0.1):
    H, W, C = X.shape

    # SLIC on first 3 PCs, scaled to [0,1], no RGB->LAB conversion
    rgb = X[:, :, :3]
    rgb = (rgb - rgb.min(axis=(0, 1))) / (rgb.max(axis=(0, 1)) - rgb.min(axis=(0, 1)) + 1e-8)
    segments = slic(rgb, n_segments=n_segments, compactness=compactness,
                    convert2lab=False, channel_axis=-1, start_label=0)

    # make ids contiguous 0..N-1 (SLIC can return fewer segments than requested)
    _, segments = np.unique(segments, return_inverse=True)
    segments = segments.reshape(H, W)
    N = segments.max() + 1

    # node feature = mean 30-d PCA vector of the superpixel
    flat_seg = segments.ravel()
    counts = np.bincount(flat_seg, minlength=N).astype(np.float32)
    feats = np.zeros((N, C), dtype=np.float32)
    for c in range(C):
        feats[:, c] = np.bincount(flat_seg, weights=X[:, :, c].ravel(), minlength=N) / counts

    # KNN graph on node features (Euclidean), symmetrised, + self loops, D^-1/2 A D^-1/2
    A = kneighbors_graph(feats, k, mode="connectivity", include_self=False).toarray()
    A = np.maximum(A, A.T) + np.eye(N)
    d_inv_sqrt = 1.0 / np.sqrt(A.sum(axis=1))
    A_hat = A * d_inv_sqrt[:, None] * d_inv_sqrt[None, :]

    return (torch.tensor(feats).float(), torch.tensor(A_hat).float(),
            torch.tensor(segments).long(), N)

node_feats, A_hat, seg_map, N_NODES = build_graph(X_pca, N_SEGMENTS, KNN_K, COMPACTNESS)
node_feats, A_hat = node_feats.to(device), A_hat.to(device)
print("Graph nodes:", N_NODES, "| node features:", tuple(node_feats.shape), "| adjacency:", tuple(A_hat.shape))

In [ ]:
# CELL 5: 70/30 STRATIFIED SPLIT (on labelled pixels, BEFORE patch extraction) + PATCH DATASET
rows, cols = np.nonzero(gt)
labels = gt[rows, cols] - 1                      # 0..num_classes-1

all_idx = np.arange(len(labels))
train_idx, test_idx = train_test_split(all_idx, test_size=0.3, stratify=labels, random_state=SEED)
print("Train:", len(train_idx), "| Test:", len(test_idx))

pad = PATCH // 2
X_pad = np.pad(X_pca, ((pad, pad), (pad, pad), (0, 0)), mode="reflect")

class PatchDataset(Dataset):
    # returns (7x7xC patch as C x 7 x 7, superpixel id of the centre pixel, label)
    def __init__(self, idx):
        self.idx = idx
    def __len__(self):
        return len(self.idx)
    def __getitem__(self, i):
        j = self.idx[i]
        r, c = rows[j], cols[j]
        patch = torch.from_numpy(X_pad[r:r + PATCH, c:c + PATCH]).permute(2, 0, 1)
        return patch, seg_map[r, c], torch.tensor(labels[j])

train_loader = DataLoader(PatchDataset(train_idx), batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(PatchDataset(test_idx),  batch_size=256, shuffle=False)

In [ ]:
# CELL 6: MODEL (ViT branch + GCN/Graph-Transformer branch + fusion)
class PatchEmbedding(nn.Module):
    def __init__(self, in_channels, embed_dim, n_tokens=PATCH * PATCH):
        super().__init__()
        self.proj = nn.Conv2d(in_channels, embed_dim, 3, padding=1)
        self.pos = nn.Parameter(torch.randn(1, n_tokens, embed_dim) * 0.02)

    def forward(self, x):                                   # (B, C, 7, 7)
        x = self.proj(x).flatten(2).transpose(1, 2)         # (B, 49, dim)
        return x + self.pos

class ViTEncoder(nn.Module):
    def __init__(self, dim, heads=4, layers=2):
        super().__init__()
        layer = nn.TransformerEncoderLayer(d_model=dim, nhead=heads, dim_feedforward=dim * 2,
                                           dropout=0.1, batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers=layers, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(dim)

    def forward(self, x):
        return self.norm(self.encoder(x)).mean(dim=1)       # (B, dim)

class GCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim)

    def forward(self, X, A_hat):                            # A_hat is pre-normalised
        return F.relu(self.linear(A_hat @ X))

class GraphTransformer(nn.Module):
    def __init__(self, dim, heads=4):
        super().__init__()
        self.attn = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.norm1 = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(nn.Linear(dim, dim * 2), nn.ReLU(), nn.Linear(dim * 2, dim))
        self.norm2 = nn.LayerNorm(dim)

    def forward(self, x):                                   # x: (N_nodes, dim)
        x = x.unsqueeze(0)
        a, _ = self.attn(x, x, x)                           # attention over all nodes
        x = self.norm1(x + a)
        x = self.norm2(x + self.ffn(x))
        return x.squeeze(0)                                 # (N_nodes, dim)

class HSIModel(nn.Module):
    def __init__(self, in_channels, vit_dim, graph_dim, num_classes):
        super().__init__()
        self.patch_embed = PatchEmbedding(in_channels, vit_dim)
        self.vit = ViTEncoder(vit_dim)
        self.gcn1 = GCNLayer(in_channels, graph_dim)
        self.gcn2 = GCNLayer(graph_dim, graph_dim)
        self.graph_trans = GraphTransformer(graph_dim)
        self.fusion = nn.Sequential(nn.Linear(vit_dim + graph_dim, 128), nn.ReLU(), nn.Dropout(0.3))
        self.cls = nn.Linear(128, num_classes)

    def forward(self, patch, seg_ids, node_feats, A_hat):
        vit_feat = self.vit(self.patch_embed(patch))                    # (B, 128) pixel-level
        g = self.gcn2(self.gcn1(node_feats, A_hat), A_hat)              # (N, 64)
        g = self.graph_trans(g)                                         # (N, 64) region-level, computed once per batch
        graph_feat = g[seg_ids]                                         # (B, 64) superpixel of each pixel
        fused = self.fusion(torch.cat([vit_feat, graph_feat], dim=-1))
        return self.cls(fused)

model = HSIModel(N_PCA, VIT_DIM, GRAPH_DIM, num_classes).to(device)
print("Parameters:", sum(p.numel() for p in model.parameters()))

In [ ]:
# CELL 7: TRAINING (Adam lr=0.001, 15 epochs, class-weighted CrossEntropy)
class_counts = np.bincount(labels[train_idx], minlength=num_classes)
weights = len(train_idx) / (num_classes * np.maximum(class_counts, 1))   # inverse frequency, mean ~ 1
loss_fn = nn.CrossEntropyLoss(weight=torch.tensor(weights, dtype=torch.float32).to(device))
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

for epoch in range(EPOCHS):
    model.train()
    total_loss, correct, seen = 0.0, 0, 0
    for patch, seg_ids, y in train_loader:
        patch, seg_ids, y = patch.to(device), seg_ids.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(patch, seg_ids, node_feats, A_hat)
        loss = loss_fn(out, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(y)
        correct += (out.argmax(1) == y).sum().item()
        seen += len(y)
    print(f"Epoch {epoch + 1:02d}/{EPOCHS} | Loss: {total_loss / seen:.4f} | Train acc: {correct / seen:.4f}")

In [ ]:
# CELL 8: EVALUATION (OA, AA, Kappa, per-class precision / recall / F1)
@torch.no_grad()
def predict(loader):
    model.eval()
    preds, trues = [], []
    for patch, seg_ids, y in loader:
        out = model(patch.to(device), seg_ids.to(device), node_feats, A_hat)
        preds.append(out.argmax(1).cpu().numpy())
        trues.append(y.numpy())
    return np.concatenate(trues), np.concatenate(preds)

y_true, y_pred = predict(test_loader)

names = CLASS_NAMES.get(DATASET, [str(i + 1) for i in range(num_classes)])
if len(names) != num_classes:
    names = [str(i + 1) for i in range(num_classes)]
class_ids = list(range(num_classes))

prec, rec, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=class_ids, zero_division=0)

OA    = accuracy_score(y_true, y_pred)
AA    = rec.mean()                                   # mean of per-class recall
KAPPA = cohen_kappa_score(y_true, y_pred)

print("===== Metrics =====")
print(f"OA    : {OA * 100:.2f}")
print(f"AA    : {AA * 100:.2f}")
print(f"Kappa : {KAPPA * 100:.2f}")
print()
print(classification_report(y_true, y_pred, labels=class_ids, target_names=names, zero_division=0, digits=4))

In [ ]:
# CELL 9: FIGURES: per-class P/R/F1 bar chart + confusion matrix + top confusions
x = np.arange(num_classes)
plt.figure(figsize=(14, 6))
plt.bar(x - 0.25, prec, width=0.25, label="Precision")
plt.bar(x,        rec,  width=0.25, label="Recall")
plt.bar(x + 0.25, f1,   width=0.25, label="F1-Score")
plt.xticks(x, names, rotation=60, ha="right")
plt.ylabel("Score"); plt.title("Per-Class Performance (Precision, Recall, F1-score)")
plt.legend(); plt.grid(axis="y", alpha=0.3); plt.tight_layout(); plt.show()

cm = confusion_matrix(y_true, y_pred, labels=class_ids)
cm_norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
plt.figure(figsize=(10, 8))
plt.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
plt.colorbar(label="Row-normalised")
plt.xticks(x, names, rotation=90); plt.yticks(x, names)
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("Confusion Matrix")
plt.tight_layout(); plt.show()

# most frequent mistakes (e.g. Grapes_untrained -> Lettuce_romaine_7wk)
off = cm.copy(); np.fill_diagonal(off, 0)
print("Top confusions (true -> predicted : count)")
for flat_i in np.argsort(off.ravel())[::-1][:5]:
    t, p = divmod(flat_i, num_classes)
    if off[t, p] > 0:
        print(f"  {names[t]} -> {names[p]} : {off[t, p]}")

In [ ]:
# CELL 10: CROP MAP (ground truth vs prediction on all labelled pixels)
full_loader = DataLoader(PatchDataset(np.arange(len(labels))), batch_size=256, shuffle=False)
_, full_pred = predict(full_loader)

pred_map = np.zeros((H, W), dtype=np.int64)
pred_map[rows, cols] = full_pred + 1

fig, ax = plt.subplots(1, 2, figsize=(12, 7))
for a, m, t in zip(ax, [gt, pred_map], ["Ground Truth", "Predicted Crop Map"]):
    a.imshow(np.ma.masked_equal(m, 0), cmap="nipy_spectral", vmin=1, vmax=num_classes)
    a.set_title(t); a.axis("off")
plt.tight_layout(); plt.show()